# Combined Corpus Creation for Model Training

This notebook creates training-ready corpus files by:
1. Processing JSON files from tripitaka.online
2. Combining with existing PDF corpus
3. **Splitting all text into sentences (one per line)**
4. Creating train/validation/test splits

## Output Format:
- **One sentence per line** - ideal for language model training
- Both PDF and web-scraped data combined
- Clean train/val/test splits for each language

## Output Structure:
```
data/corpus_training/
├── splits/
│   ├── sinhala/
│   │   ├── train.txt         # One sentence per line
│   │   ├── validation.txt
│   │   └── test.txt
│   ├── pali/
│   │   ├── train.txt
│   │   ├── validation.txt
│   │   └── test.txt
│   └── mixed/
│       ├── train.txt
│       ├── validation.txt
│       └── test.txt
└── metadata/
    ├── corpus_stats.json
    └── sentence_counts.json
```

## 1. Setup and Installation

In [1]:
!pip install -q tqdm pandas

In [2]:
import os
import re
import json
import random
from pathlib import Path
from typing import Dict, List, Tuple
from datetime import datetime
from collections import defaultdict

import pandas as pd
from tqdm.auto import tqdm

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


/Users/ranidugurusinghe/Home/Business/Upwork/Projects/Shrey/congenial-journey/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Mount Google Drive (if using Colab)
try:
    # from google.colab import drive
    # drive.mount('/content/drive')
    print("✓ Google Drive mounted")
except:
    print("ℹ Running locally (not on Colab)")

Mounted at /content/drive
✓ Google Drive mounted


## 2. Configuration

In [15]:
# ===== CONFIGURE THESE PATHS =====
# Base project directory
PROJECT_DIR = Path("/Users/ranidugurusinghe/Home/University/Assignments/Year 4/Final Year Project/Multi-lingual-Buddhist-Conversational-Agent/")

# Input directories
JSON_INPUT_DIR = PROJECT_DIR / "data" / "5-webscrapping-tripitaka"
PDF_CORPUS_DIR = PROJECT_DIR / "data" / "4-data-cleaning-corpus-creation" / "raw" / "by_language"

# Add this with the other input paths
PDF_CORPUS_CSV_DIR = PROJECT_DIR / "data" / "4.3-corpus" / "structured"

# Output directory for training corpus
TRAINING_CORPUS_DIR = PROJECT_DIR / "data" / "corpus_training"
SPLITS_DIR = TRAINING_CORPUS_DIR / "splits"
METADATA_DIR = TRAINING_CORPUS_DIR / "metadata"

# Create directories
for lang in ['sinhala', 'pali', 'mixed']:
    (SPLITS_DIR / lang).mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# Verify paths
print(f"✓ Project: {PROJECT_DIR}")
print(f"✓ JSON input: {JSON_INPUT_DIR}")
print(f"✓ PDF corpus: {PDF_CORPUS_DIR}")
print(f"✓ Training output: {TRAINING_CORPUS_DIR}")

✓ Project: /Users/ranidugurusinghe/Home/University/Assignments/Year 4/Final Year Project/Multi-lingual-Buddhist-Conversational-Agent
✓ JSON input: /Users/ranidugurusinghe/Home/University/Assignments/Year 4/Final Year Project/Multi-lingual-Buddhist-Conversational-Agent/data/5-webscrapping-tripitaka
✓ PDF corpus: /Users/ranidugurusinghe/Home/University/Assignments/Year 4/Final Year Project/Multi-lingual-Buddhist-Conversational-Agent/data/4-data-cleaning-corpus-creation/raw/by_language
✓ Training output: /Users/ranidugurusinghe/Home/University/Assignments/Year 4/Final Year Project/Multi-lingual-Buddhist-Conversational-Agent/data/corpus_training


In [16]:
# Configuration
MIN_TEXT_LENGTH = 50
MIN_SENTENCE_LENGTH = 10  # Minimum characters for a valid sentence

SPLIT_RATIOS = {
    'train': 0.8,
    'validation': 0.1,
    'test': 0.1
}

NIKAYAS = ['digha', 'majjhima', 'samyutta', 'anguttara', 'khuddaka']
RANDOM_SEED = 42

print(f"Configuration:")
print(f"  Split ratios: {SPLIT_RATIOS}")
print(f"  Min sentence length: {MIN_SENTENCE_LENGTH} chars")
print(f"  Random seed: {RANDOM_SEED}")

Configuration:
  Split ratios: {'train': 0.8, 'validation': 0.1, 'test': 0.1}
  Min sentence length: 10 chars
  Random seed: 42


## 3. Sentence Splitting Functions

In [17]:
def split_into_sentences(text: str, min_length: int = 10) -> List[str]:
    """
    Split text into sentences for Sinhala and Pali.
    Handles both Sinhala (|, ., ?, !) and Pali (. !) sentence boundaries.
    """
    # Split on common sentence delimiters
    # Sinhala uses | (pipe) and . period
    # Pali typically uses . period
    # Also handle ?, ! and newlines

    # First normalize multiple spaces/newlines
    text = re.sub(r'\s+', ' ', text)

    # Split on sentence boundaries
    # Pattern: pipe, period, question mark, exclamation, or double newline
    sentences = re.split(r'[|.?!]+|\n\n+', text)

    # Clean and filter sentences
    cleaned_sentences = []
    for sent in sentences:
        # Strip whitespace
        sent = sent.strip()

        # Skip empty or too short
        if len(sent) < min_length:
            continue

        # Skip if only punctuation/numbers
        if re.match(r'^[\d\s.,;:!?()\[\]{}"\'-]+$', sent):
            continue

        cleaned_sentences.append(sent)

    return cleaned_sentences

def test_sentence_splitter():
    """Test the sentence splitter"""
    test_sinhala = "මා හට අසන්නට ලැබුණේ මේ විදිහටයි| භාග්‍යවත් බුදුරජාණන් වහන්සේ රාජගහ නුවරට සැරිසරන විටයි. එතුමන් ආරාමයක වැඩ වසන සේක."
    test_pali = "Evaṃ me sutaṃ. Ekaṃ samayaṃ bhagavā antarā ca rājagahaṃ antarā ca nāḷandaṃ."

    sinhala_sents = split_into_sentences(test_sinhala)
    pali_sents = split_into_sentences(test_pali)

    print("Test Results:")
    print(f"\nSinhala ({len(sinhala_sents)} sentences):")
    for i, s in enumerate(sinhala_sents, 1):
        print(f"  {i}. {s[:80]}..." if len(s) > 80 else f"  {i}. {s}")

    print(f"\nPali ({len(pali_sents)} sentences):")
    for i, s in enumerate(pali_sents, 1):
        print(f"  {i}. {s[:80]}..." if len(s) > 80 else f"  {i}. {s}")

# Run test
test_sentence_splitter()
print("\n✓ Sentence splitter loaded and tested")

Test Results:

Sinhala (3 sentences):
  1. මා හට අසන්නට ලැබුණේ මේ විදිහටයි
  2. භාග්‍යවත් බුදුරජාණන් වහන්සේ රාජගහ නුවරට සැරිසරන විටයි
  3. එතුමන් ආරාමයක වැඩ වසන සේක

Pali (2 sentences):
  1. Evaṃ me sutaṃ
  2. Ekaṃ samayaṃ bhagavā antarā ca rājagahaṃ antarā ca nāḷandaṃ

✓ Sentence splitter loaded and tested


## 4. Load and Process JSON Files

In [18]:
def load_json_file(filepath: Path) -> List[Dict]:
    """Load JSON file"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return [data] if isinstance(data, dict) else data
    except Exception as e:
        print(f"Error loading {filepath}: {e}")
        return []

def extract_sentences_from_json(json_data: List[Dict], nikaya: str, min_sent_len: int) -> Tuple[List[Dict], List[Dict]]:
    """
    Extract sentences from JSON files with metadata attached.
    Returns: (sinhala_records, pali_records)
    Each record is a dict with text, book_name_en, book_name_si, source.
    """
    sinhala_records = []
    pali_records = []

    for entry in json_data:
        if not entry.get('is_valid_content', False):
            continue

        content = entry.get('content', {})

        if 'sinhala' in content and content['sinhala']:
            sinhala_text = content['sinhala']
            if len(sinhala_text) >= MIN_TEXT_LENGTH:
                for sent in split_into_sentences(sinhala_text, min_sent_len):
                    sinhala_records.append({
                        'text': sent,
                        'language': 'sinhala',
                        'book_name_en': f"{nikaya.capitalize()} Nikaya",
                        'book_name_si': '',
                        'source': 'Tripitaka'
                    })

        if 'pali' in content and content['pali']:
            pali_text = content['pali']
            if len(pali_text) >= MIN_TEXT_LENGTH:
                for sent in split_into_sentences(pali_text, min_sent_len):
                    pali_records.append({
                        'text': sent,
                        'language': 'pali',
                        'book_name_en': f"{nikaya.capitalize()} Nikaya",
                        'book_name_si': '',
                        'source': 'Tripitaka'
                    })

    return sinhala_records, pali_records

print("✓ JSON processing functions loaded")

✓ JSON processing functions loaded


In [19]:
# Process all JSON files
print("Processing JSON files from tripitaka.online...\n")

all_json_sinhala_records = []
all_json_pali_records = []
json_stats = defaultdict(lambda: {'files': 0, 'sinhala_sents': 0, 'pali_sents': 0})

for nikaya in NIKAYAS:
    nikaya_dir = JSON_INPUT_DIR / nikaya

    if not nikaya_dir.exists():
        print(f"⚠ {nikaya} directory not found")
        continue

    json_files = list(nikaya_dir.glob('*.json'))

    if not json_files:
        print(f"⚠ No JSON files in {nikaya}")
        continue

    nikaya_sinhala = []
    nikaya_pali = []

    for json_file in tqdm(json_files, desc=f"  {nikaya}"):
        data = load_json_file(json_file)
        sinhala_recs, pali_recs = extract_sentences_from_json(data, nikaya, MIN_SENTENCE_LENGTH)
        nikaya_sinhala.extend(sinhala_recs)
        nikaya_pali.extend(pali_recs)

    json_stats[nikaya] = {
        'files': len(json_files),
        'sinhala_sents': len(nikaya_sinhala),
        'pali_sents': len(nikaya_pali)
    }

    all_json_sinhala_records.extend(nikaya_sinhala)
    all_json_pali_records.extend(nikaya_pali)

    print(f"  ✓ {nikaya}: {len(nikaya_sinhala):,} Sinhala, {len(nikaya_pali):,} Pali sentences")

print(f"\n{'='*70}")
print(f"JSON Processing Complete:")
print(f"  Total Sinhala sentences: {len(all_json_sinhala_records):,}")
print(f"  Total Pali sentences: {len(all_json_pali_records):,}")
print(f"{'='*70}")

Processing JSON files from tripitaka.online...



  digha: 100%|██████████| 5/5 [00:00<00:00,  8.80it/s]


  ✓ digha: 57,227 Sinhala, 29,818 Pali sentences


  majjhima: 100%|██████████| 7/7 [00:00<00:00,  8.26it/s]


  ✓ majjhima: 87,108 Sinhala, 45,804 Pali sentences


...

  ✓ samyutta: 1,822 Sinhala, 593 Pali sentences


  anguttara: 100%|██████████| 92/92 [00:01<00:00, 65.47it/s]


  ✓ anguttara: 140,987 Sinhala, 68,990 Pali sentences


  khuddaka: 100%|██████████| 46/46 [00:00<00:00, 71.98it/s]

  ✓ khuddaka: 66,763 Sinhala, 40,214 Pali sentences

JSON Processing Complete:
  Total Sinhala sentences: 353,907
  Total Pali sentences: 185,419


## 5. Load and Process PDF Corpus

In [20]:
def load_sentences_from_corpus(corpus_dir: Path, lang: str, min_sent_len: int) -> List[Dict]:
    """
    Load all text files from PDF corpus, extract metadata from folder path,
    and split into sentence-level records.

    Folder structure: by_language/{lang}/{book_category}/{book_name_si}/page_XXX.txt

    Returns list of dicts with:
    sentence_id, book_category, book_name_si, book_name_en, source, text, language
    """
    lang_dir = corpus_dir / lang

    if not lang_dir.exists():
        print(f"  ⚠ {lang} directory not found")
        return []

    text_files = list(lang_dir.rglob('*.txt'))

    if not text_files:
        print(f"  ⚠ No text files in {lang}")
        return []

    all_records = []

    for text_file in tqdm(text_files, desc=f"  Loading {lang}"):
        try:
            # Extract metadata from folder path
            # Path: by_language/{lang}/{book_category}/{book_name_si}/page_XXX.txt
            parts = text_file.relative_to(lang_dir).parts
            book_category = parts[0] if len(parts) > 2 else ''
            book_name_si = parts[1] if len(parts) > 2 else ''

            with open(text_file, 'r', encoding='utf-8') as f:
                text = f.read()

            sentences = split_into_sentences(text, min_sent_len)

            for sent in sentences:
                all_records.append({
                    'book_category': book_category,
                    'book_name_si': book_name_si,
                    'book_name_en': '',
                    'source': 'IFBC',
                    'text': sent,
                    'language': lang
                })

        except Exception as e:
            print(f"    Error reading {text_file}: {e}")

    return all_records

print("✓ PDF corpus processing function loaded")

✓ PDF corpus processing function loaded


In [23]:
# Process PDF corpus
print("\nProcessing PDF corpus...\n")

pdf_records = {}

for lang in ['sinhala', 'mixed']:
    records = load_sentences_from_corpus(PDF_CORPUS_DIR, lang, MIN_SENTENCE_LENGTH)
    pdf_records[lang] = records
    print(f"  ✓ {lang}: {len(records):,} sentence records")

print(f"\n{'='*70}")
print(f"PDF Corpus Processing Complete")
print(f"{'='*70}")


Processing PDF corpus...



  Loading sinhala: 100%|██████████| 6184/6184 [00:02<00:00, 2462.40it/s]


  ✓ sinhala: 111,632 sentence records


  Loading mixed: 100%|██████████| 7066/7066 [00:02<00:00, 2673.24it/s]

  ✓ mixed: 135,386 sentence records

PDF Corpus Processing Complete


## 6. Combine All Sentences

In [25]:
print("\nBuilding combined records and saving splits...\n")

# Build Tripitaka records with matching column structure
def build_tripitaka_records(json_records: List[Dict]) -> List[Dict]:
    """Convert JSON records to match the unified column structure."""
    result = []
    for r in json_records:
        result.append({
            'book_category': 'tripitaka',
            'book_name_si': '',
            'book_name_en': r['book_name_en'],
            'source': 'Tripitaka',
            'text': r['text'],
            'language': r['language']
        })
    return result

tripitaka_sinhala_records = build_tripitaka_records(all_json_sinhala_records)
tripitaka_pali_records = build_tripitaka_records(all_json_pali_records)

print(f"  ✓ Tripitaka Sinhala records: {len(tripitaka_sinhala_records):,}")
print(f"  ✓ Tripitaka Pali records: {len(tripitaka_pali_records):,}")

# Combine per language
# sinhala = IFBC sinhala + Tripitaka sinhala
# pali    = IFBC pali only
# mixed   = IFBC mixed + Tripitaka pali
combined_records = {
    'sinhala': pdf_records['sinhala'] + tripitaka_sinhala_records,
    'mixed':   pdf_records['mixed'] + tripitaka_pali_records
}

print(f"\n  Combined totals:")
for lang, recs in combined_records.items():
    print(f"    {lang}: {len(recs):,} sentences")

# Assign global sentence IDs across all languages
print("\n  Assigning sentence IDs...")
global_counter = 1
for lang in ['sinhala', 'mixed']:
    for rec in combined_records[lang]:
        rec['sentence_id'] = global_counter
        global_counter += 1

print(f"  ✓ Total sentences assigned IDs: {global_counter - 1:,}")

# Define column order
COLUMNS = ['sentence_id', 'book_category', 'book_name_si', 'book_name_en', 'source', 'text', 'language']

# Shuffle, split and save both CSV and txt per language
print("\nSaving splits (txt + csv)...\n")

split_stats = {}

for lang in ['sinhala', 'mixed']:
    records = combined_records[lang]

    if not records:
        print(f"  ⚠ {lang}: no records, skipping")
        continue

    # Shuffle
    random.seed(RANDOM_SEED)
    shuffled = records.copy()
    random.shuffle(shuffled)

    # Split
    n = len(shuffled)
    train_end = int(n * SPLIT_RATIOS['train'])
    val_end = train_end + int(n * SPLIT_RATIOS['validation'])

    splits = {
        'train':      shuffled[:train_end],
        'validation': shuffled[train_end:val_end],
        'test':       shuffled[val_end:]
    }

    lang_dir = SPLITS_DIR / lang
    lang_dir.mkdir(parents=True, exist_ok=True)

    split_stats[lang] = {'total_sentences': n}

    for split_name, split_records in splits.items():
        split_stats[lang][split_name] = len(split_records)

        # Save txt (one sentence per line, for model training)
        txt_path = lang_dir / f"{split_name}.txt"
        with open(txt_path, 'w', encoding='utf-8') as f:
            for rec in split_records:
                f.write(rec['text'] + '\n')

        # Save csv (with full metadata, for HuggingFace)
        csv_path = lang_dir / f"{split_name}.csv"
        df = pd.DataFrame(split_records, columns=COLUMNS)
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')

        print(f"  ✓ {lang}/{split_name}: {len(split_records):,} sentences → .txt + .csv saved")

    print()

print(f"{'='*70}")
print("✓ ALL SPLITS SAVED")
print(f"{'='*70}")


Building combined records and saving splits...

  ✓ Tripitaka Sinhala records: 353,907
  ✓ Tripitaka Pali records: 185,419

  Combined totals:
    sinhala: 465,539 sentences
    mixed: 320,805 sentences

  Assigning sentence IDs...
  ✓ Total sentences assigned IDs: 786,344

Saving splits (txt + csv)...

  ✓ sinhala/train: 372,431 sentences → .txt + .csv saved
  ✓ sinhala/validation: 46,553 sentences → .txt + .csv saved
  ✓ sinhala/test: 46,555 sentences → .txt + .csv saved

  ✓ mixed/train: 256,644 sentences → .txt + .csv saved
  ✓ mixed/validation: 32,080 sentences → .txt + .csv saved
  ✓ mixed/test: 32,081 sentences → .txt + .csv saved

✓ ALL SPLITS SAVED


In [26]:
# Build combined_sentences from records for statistics display
combined_sentences = {
    lang: [r['text'] for r in recs]
    for lang, recs in combined_records.items()
}

print("Combined Sentence Counts:")
for lang, sents in combined_sentences.items():
    pdf_count = len(pdf_records[lang])
    json_count = len(sents) - pdf_count
    print(f"\n{lang.upper()}:")
    print(f"  IFBC corpus: {pdf_count:,} sentences")
    if lang == 'sinhala':
        print(f"  Tripitaka (Sinhala): {json_count:,} sentences")
    elif lang == 'mixed':
        print(f"  Tripitaka (Pali): {json_count:,} sentences")
    print(f"  TOTAL: {len(sents):,} sentences")

print(f"\n{'='*70}")

Combined Sentence Counts:

SINHALA:
  IFBC corpus: 111,632 sentences
  Tripitaka (Sinhala): 353,907 sentences
  TOTAL: 465,539 sentences

MIXED:
  IFBC corpus: 135,386 sentences
  Tripitaka (Pali): 185,419 sentences
  TOTAL: 320,805 sentences



## 7. Create Train/Validation/Test Splits

In [12]:
def create_splits(sentences: List[str], split_ratios: Dict, random_seed: int = 42) -> Dict[str, List[str]]:
    """
    Split sentences into train/validation/test sets.
    """
    # Shuffle sentences
    random.seed(random_seed)
    shuffled = sentences.copy()
    random.shuffle(shuffled)

    # Calculate split points
    n = len(shuffled)
    train_end = int(n * split_ratios['train'])
    val_end = train_end + int(n * split_ratios['validation'])

    return {
        'train': shuffled[:train_end],
        'validation': shuffled[train_end:val_end],
        'test': shuffled[val_end:]
    }

def save_split(sentences: List[str], filepath: Path):
    """
    Save sentences to file (one per line).
    """
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        for sent in sentences:
            f.write(sent + '\n')

print("✓ Split creation functions loaded")

✓ Split creation functions loaded


In [13]:
# Create and save splits for each language
print("\nCreating train/validation/test splits...\n")

split_stats = {}

for lang, sentences in combined_sentences.items():
    print(f"Processing {lang}...")

    # Create splits
    splits = create_splits(sentences, SPLIT_RATIOS, RANDOM_SEED)

    # Save each split
    lang_dir = SPLITS_DIR / lang

    for split_name, split_sentences in splits.items():
        filepath = lang_dir / f"{split_name}.txt"
        save_split(split_sentences, filepath)
        print(f"  ✓ {split_name}: {len(split_sentences):,} sentences → {filepath.name}")

    # Store stats
    split_stats[lang] = {
        'total_sentences': len(sentences),
        'train': len(splits['train']),
        'validation': len(splits['validation']),
        'test': len(splits['test'])
    }

    print()

print(f"{'='*70}")
print("Splits created successfully!")
print(f"{'='*70}")


Creating train/validation/test splits...

Processing sinhala...
  ✓ train: 372,431 sentences → train.txt
  ✓ validation: 46,553 sentences → validation.txt
  ✓ test: 46,555 sentences → test.txt

Processing pali...
  ✓ train: 396 sentences → train.txt
  ✓ validation: 49 sentences → validation.txt
  ✓ test: 50 sentences → test.txt

Processing mixed...
  ✓ train: 256,644 sentences → train.txt
  ✓ validation: 32,080 sentences → validation.txt
  ✓ test: 32,081 sentences → test.txt

Splits created successfully!


## 8. Generate Metadata and Statistics

In [14]:
# Compile comprehensive statistics
corpus_stats = {
    'created_at': datetime.now().isoformat(),
    'random_seed': RANDOM_SEED,
    'split_ratios': SPLIT_RATIOS,
    'min_sentence_length': MIN_SENTENCE_LENGTH,

    'sources': {
        'pdf_corpus': {
            'path': str(PDF_CORPUS_DIR),
            'sinhala_sentences': len(pdf_sentences['sinhala']),
            'pali_sentences': len(pdf_sentences['pali']),
            'mixed_sentences': len(pdf_sentences['mixed'])
        },
        'tripitaka_online': {
            'path': str(JSON_INPUT_DIR),
            'sinhala_sentences': len(all_json_sinhala_sents),
            'pali_sentences': len(all_json_pali_sents),
            'by_nikaya': dict(json_stats)
        }
    },

    'combined_corpus': {
        'sinhala': {
            'sources': ['PDF corpus', 'Tripitaka.online Sinhala'],
            'total_sentences': len(combined_sentences['sinhala']),
            'pdf_sentences': len(pdf_sentences['sinhala']),
            'json_sentences': len(all_json_sinhala_sents)
        },
        'pali': {
            'sources': ['PDF corpus'],
            'total_sentences': len(combined_sentences['pali']),
            'pdf_sentences': len(pdf_sentences['pali'])
        },
        'mixed': {
            'sources': ['PDF corpus', 'Tripitaka.online Pali'],
            'total_sentences': len(combined_sentences['mixed']),
            'pdf_sentences': len(pdf_sentences['mixed']),
            'json_sentences': len(all_json_pali_sents)
        }
    },

    'splits': split_stats,

    'output_directory': str(TRAINING_CORPUS_DIR)
}

# Save statistics
stats_file = METADATA_DIR / "corpus_stats.json"
with open(stats_file, 'w', encoding='utf-8') as f:
    json.dump(corpus_stats, f, ensure_ascii=False, indent=2)

print(f"\n✓ Statistics saved to {stats_file}")


✓ Statistics saved to /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/corpus_training/metadata/corpus_stats.json


In [15]:
# Create detailed sentence counts
sentence_counts = {
    'by_language': {},
    'by_split': {}
}

for lang in ['sinhala', 'pali', 'mixed']:
    sentence_counts['by_language'][lang] = {
        'total': len(combined_sentences[lang]),
        'train': split_stats[lang]['train'],
        'validation': split_stats[lang]['validation'],
        'test': split_stats[lang]['test']
    }

for split in ['train', 'validation', 'test']:
    sentence_counts['by_split'][split] = {
        'sinhala': split_stats['sinhala'][split],
        'pali': split_stats['pali'][split],
        'mixed': split_stats['mixed'][split],
        'total': sum(split_stats[lang][split] for lang in ['sinhala', 'pali', 'mixed'])
    }

counts_file = METADATA_DIR / "sentence_counts.json"
with open(counts_file, 'w', encoding='utf-8') as f:
    json.dump(sentence_counts, f, ensure_ascii=False, indent=2)

print(f"✓ Sentence counts saved to {counts_file}")

✓ Sentence counts saved to /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/corpus_training/metadata/sentence_counts.json


## 9. Final Summary

In [16]:
print("\n" + "="*80)
print("TRAINING CORPUS CREATION COMPLETE")
print("="*80)

print("\n📊 FINAL STATISTICS:\n")

# By language
print("By Language:")
for lang in ['sinhala', 'pali', 'mixed']:
    stats = split_stats[lang]
    print(f"\n  {lang.upper()}:")
    print(f"    Total: {stats['total_sentences']:,} sentences")
    print(f"    Train: {stats['train']:,} ({stats['train']/stats['total_sentences']*100:.1f}%)")
    print(f"    Val:   {stats['validation']:,} ({stats['validation']/stats['total_sentences']*100:.1f}%)")
    print(f"    Test:  {stats['test']:,} ({stats['test']/stats['total_sentences']*100:.1f}%)")

# By split
print("\nBy Split:")
for split in ['train', 'validation', 'test']:
    total = sum(split_stats[lang][split] for lang in ['sinhala', 'pali', 'mixed'])
    print(f"  {split.capitalize()}: {total:,} total sentences")

# Grand total
grand_total = sum(stats['total_sentences'] for stats in split_stats.values())
print(f"\n  GRAND TOTAL: {grand_total:,} sentences")

print("\n📁 OUTPUT STRUCTURE:")
print(f"  {TRAINING_CORPUS_DIR}/")
print(f"  ├── splits/")
for lang in ['sinhala', 'pali', 'mixed']:
    print(f"  │   ├── {lang}/")
    print(f"  │   │   ├── train.txt")
    print(f"  │   │   ├── validation.txt")
    print(f"  │   │   └── test.txt")
print(f"  └── metadata/")
print(f"      ├── corpus_stats.json")
print(f"      └── sentence_counts.json")

print("\n✅ READY FOR TRAINING!")
print("\nFormat: One sentence per line in each split file")
print("Use these files for:")
print("  • Language model pretraining")
print("  • Fine-tuning")
print("  • Tokenizer training")
print("  • Evaluation benchmarks")

print("\n" + "="*80)


TRAINING CORPUS CREATION COMPLETE

📊 FINAL STATISTICS:

By Language:

  SINHALA:
    Total: 465,539 sentences
    Train: 372,431 (80.0%)
    Val:   46,553 (10.0%)
    Test:  46,555 (10.0%)

  PALI:
    Total: 495 sentences
    Train: 396 (80.0%)
    Val:   49 (9.9%)
    Test:  50 (10.1%)

  MIXED:
    Total: 320,805 sentences
    Train: 256,644 (80.0%)
    Val:   32,080 (10.0%)
    Test:  32,081 (10.0%)

By Split:
  Train: 629,471 total sentences
  Validation: 78,682 total sentences
  Test: 78,686 total sentences

  GRAND TOTAL: 786,839 sentences

📁 OUTPUT STRUCTURE:
  /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/corpus_training/
  ├── splits/
  │   ├── sinhala/
  │   │   ├── train.txt
  │   │   ├── validation.txt
  │   │   └── test.txt
  │   ├── pali/
  │   │   ├── train.txt
  │   │   ├── validation.txt
  │   │   └── test.txt
  │   ├── mixed/
  │   │   ├── train.txt
  │   │   ├── validation.txt
  │   │   └── test.txt
  └── metadata/
      ├── 

## 10. Sample Verification

In [17]:
# Show sample sentences from each language
print("\n" + "="*80)
print("SAMPLE SENTENCES (First 5 from each language's training set)")
print("="*80)

for lang in ['sinhala', 'pali', 'mixed']:
    train_file = SPLITS_DIR / lang / "train.txt"

    print(f"\n{lang.upper()}:")
    with open(train_file, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            if i > 5:
                break
            sent = line.strip()
            # Truncate long sentences for display
            if len(sent) > 100:
                sent = sent[:100] + "..."
            print(f"  {i}. {sent}")

print("\n" + "="*80)


SAMPLE SENTENCES (First 5 from each language's training set)

SINHALA:
  1. සද්ධර්මරත්නාවලි ය " බොහෝ වහන්දෑ සුප්පබුද්ධ කුෂ්ඨින් මළ නියා ව බුදුන්ට දන්වා ලා ‘ස්වාමීනි, උන් මිය උප...
  2. ගියොත් බලා ගන්නත් පුළුවන්
  3. මෙ සේ විසි ආකාර වු පඨවි ධාතුව ද, දොළොස් ආකාර වූ ආපෝ ධාතුව ද, සතර ආකාර වූ තේජෝ ධාතුව ද, සයාකාර වූ වාය...
  4. සිතුල් පව් වෙත කපිට්ඨ නම් ගම අත්පල නම් තැනැත්ත හුගේ 83 ඵුස්සදේව යයි නම් ඇති පුතෙක් විය
  5. යම් වෙලාවක ඒ සඤ්ඤාව එනවා නම්, ඒ වෙලාවට ඔහු සඤ්ඤා සහිතව ඉන්නවා

PALI:
  1. ඉ හස්තීන්ගේ කණ මුල චුලොදරන- කුඩා බඩ, එනම් ඇත්තා
  2. න-සුංකම් අය කරණ තැන
  3. න - කරකැවීම, ආදියෙන් කළ හුලන් ඉරටුවක සවිකරගත් විට කරකැවේ
  4. පිප්ඵලකන - මහන නූල්, කතුර
  5. ක්රි-(චිචි, චිට+ආයති) චිච්, චිට යන ශබ්ද කරයි

MIXED:
  1. ඉ-වසඟ වූ ස්ත්රිය, වඳ දෙන
  2. සිංහබාහුට අවුරුදු දහසයක් ඉක්ම ගියේය
  3. සෙය්‍යථිදං: රාජකථං චෝරකථං මහාමත්තකථං සේනාකථං භයකථං යුද්ධකථං අන්නකථං පානකථං වත්ථකථං යානකථං සයනකථං මාල...
  4. පංචකාම සම්පත් විඳිමින් සතුටින් සිටිත්
  5. ඒ නරේන්ද්රතෙම රජරට නොයෙක් ගම් නියම්ගම් වලද නව අනූවක් අභි